# Systematic Base Model Pipelines and Optimization Experiments

In [2]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from src import base_model as utils
from src import mlflow_utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Global Pipeline Configuration Settings

In [3]:
RANDOM_STATE = 42
from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

In [4]:
# Ingest Transformed Dataset Partitions
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [5]:
# Standard Base Model Search Map Space
models_zoo = {
    "logistic_regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=2000),
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "xgboost": XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1)
}

In [6]:
# Define Constant Immutable Core Column Groups
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

## 2. Experiment 1: Primary Baseline Exploration (With Potential Feature Leakage)

In [7]:
EXP_1_NAME = "customer-churn-simple"
mlflow_utils.init_mlflow_experiment(EXP_1_NAME, DB_PATH, ARTIFACTS_DIR)

2026/06/14 19:30:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/14 19:30:01 INFO mlflow.store.db.utils: Updating database tables


In [8]:
# Configure Experiment Schema Arrays
nomod_columns_exp1 = ['HasCrCard', 'IsActiveMember', 'Complain']

preprocessor_exp1 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
        ('num', StandardScaler(), norm_std_columns),
        ('pass', 'passthrough', nomod_columns_exp1)
    ],
    remainder='drop'
)

In [9]:
# Run Pipeline Execution Training Loop
utils.train_and_log_models(
    models=models_zoo, preprocessor=preprocessor_exp1,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns_exp1
)

2026/06/14 19:30:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/14 19:30:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: logistic_regression


2026/06/14 19:30:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully processed and logged: decision_tree


2026/06/14 19:30:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/14 19:30:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: random_forest
Successfully processed and logged: xgboost


In [10]:
# Review Performance Metrics Results
mlflow_utils.get_experiment_summary(EXP_1_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc,start_time
3,c8e156171f01453dbc0d36afd4c0529b,logistic_regression,0.9980,0.990291,1.0,0.995122,0.999786,0.999112,2026-06-15 01:30:01.895000+00:00
1,e92450b0ebf54a40829002dcc8d9cb49,random_forest,0.9980,0.990291,1.0,0.995122,0.999770,0.998888,2026-06-15 01:30:07.040000+00:00
0,dabe55584e6e4bb2bf1808d2b7ac3de5,xgboost,0.9980,0.990291,1.0,0.995122,0.999597,0.997947,2026-06-15 01:30:09.229000+00:00
2,94c8f115d8994d4d8a98b0cdf01e9550,decision_tree,0.9975,0.987893,1.0,0.993910,0.998430,0.987893,2026-06-15 01:30:04.783000+00:00


> **Analysis Insight:** Perfect classification metrics suggest target leakage from the `Complain` column. Retraining the models without this feature is necessary to capture realistic real-world performance patterns.

## 3. Experiment 2: Robust Evaluation (Excluding Target Leakage Feature)

In [11]:
EXP_2_NAME = "customer-churn-simple-no-complain-feature"
mlflow_utils.init_mlflow_experiment(EXP_2_NAME, DB_PATH, ARTIFACTS_DIR)

In [12]:
# Sanitize Feature Spaces
nomod_columns_exp2 = ['HasCrCard', 'IsActiveMember']

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
        ('num', StandardScaler(), norm_std_columns),
        ('pass', 'passthrough', nomod_columns_exp2)
    ],
    remainder='drop'
)

In [13]:
# Run Sanitized Pipeline Iteration Execution
utils.train_and_log_models(
    models=models_zoo, preprocessor=preprocessor_exp2,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns_exp2
)

2026/06/14 19:30:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/14 19:30:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: logistic_regression
Successfully processed and logged: decision_tree


2026/06/14 19:30:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: random_forest


2026/06/14 19:30:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 19:30:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: xgboost


In [14]:
# Review Sanitized Performance Results
mlflow_utils.get_experiment_summary(EXP_2_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc,start_time
1,c568fbf3997f4b1da91e8d6f763e34b4,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,0.733387,2026-06-15 01:30:15.569000+00:00
0,13b22b12f9634aa3bfc3455bc3d12c42,xgboost,0.8540,0.682390,0.531863,0.597796,0.849671,0.691571,2026-06-15 01:30:17.773000+00:00
3,e0fa09e0177c4f04b5699050b0307ea7,logistic_regression,0.8530,0.743590,0.426471,0.542056,0.838354,0.649413,2026-06-15 01:30:11.519000+00:00
2,65c0356643a04f5c9b8465c6cb54ef76,decision_tree,0.7965,0.501080,0.568627,0.532721,0.711763,0.372928,2026-06-15 01:30:13.527000+00:00
